# Chapter 23: Natural Language Processing. Computing exercises

These exercises are from *Bayes, Artificial Intelligence, and Deep Learning: Foundations of Data Science* by Nick Polson and Vadim Sokolov, <https://github.com/VadimSokolov/bayes_ai_deep_learning>. The text is licensed under CC BY 4.0 and the code under the MIT License.

The code of the chapter, which several exercises refer to, is in [`code/23-nlp.py`](https://github.com/VadimSokolov/bayes_ai_deep_learning/blob/main/code/23-nlp.py).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VadimSokolov/bayes_ai_deep_learning/blob/main/exercises/23-nlp.ipynb)

In [ ]:
# Modules that the exercises of this notebook use: numpy (all installed on Colab)

## Exercise 23.7: Skip-Gram with Negative Sampling

Consider a toy corpus of sixteen three-word "sentences" drawn from two unrelated themes, royalty and pets:

In [ ]:
corpus = [
    "king queen throne", "queen king throne", "king queen palace", "queen king crown",
    "king throne crown", "queen throne palace", "king palace crown", "queen palace crown",
    "dog cat pet", "cat dog pet", "dog cat animal", "cat dog leash",
    "dog animal leash", "cat animal pet", "dog leash pet", "cat leash animal",
]
sentences = [s.split() for s in corpus]

a) Build the vocabulary and, treating each three-word sentence as its own context window (every word is a context word for every other word in the same sentence), construct the list of (center, context) index pairs used for training. Report the vocabulary size and the number of training pairs.

b) Implement skip-gram with negative sampling (Chapter 23) from scratch: a center-word embedding matrix and a context-word embedding matrix, each with one $d=8$ dimensional row per vocabulary word, a sigmoid function, and the noise distribution $P_n(w) \propto f(w)^{3/4}$ built from the unigram counts $f(w)$. Run manual stochastic gradient descent for 400 epochs with $K=8$ negative samples per positive pair and a learning rate that decays linearly over training, printing the average per-pair loss every 100 epochs. Confirm the loss decreases.

c) Using the trained center embeddings, compute the cosine similarity for the two within-theme pairs (king, queen) and (dog, cat), and for the four cross-theme pairs (king, dog), (king, cat), (queen, dog), (queen, cat). Does the model separate the two themes, even though it was never told they exist?

## Exercise 23.8: GloVe from Global Co-occurrence Counts

While skip-gram slides a window over sentences, GloVe (Chapter 23) fits word vectors directly to global co-occurrence counts. Its motivating observation is that *ratios* of co-occurrence probabilities carry meaning: $\mathrm{P}\left(\text{solid} \mid \text{ice}\right)/\mathrm{P}\left(\text{solid} \mid \text{steam}\right)$ is large, $\mathrm{P}\left(\text{gas} \mid \text{ice}\right)/\mathrm{P}\left(\text{gas} \mid \text{steam}\right)$ is small, and $\mathrm{P}\left(\text{water} \mid \text{ice}\right)/\mathrm{P}\left(\text{water} \mid \text{steam}\right)$ is near 1.

Work over the five-word vocabulary `["gas", "ice", "solid", "steam", "water"]` with symmetric co-occurrence counts ice-steam 3, ice-solid 9, ice-gas 1, ice-water 5, steam-solid 1, steam-gas 9, steam-water 5, solid-gas 1, solid-water 3, and gas-water 3 (all unlisted pairs 0).

In [ ]:
vocab = ["gas", "ice", "solid", "steam", "water"]
idx = {w: i for i, w in enumerate(vocab)}
counts = {
    ("ice", "steam"): 3, ("ice", "solid"): 9, ("ice", "gas"): 1, ("ice", "water"): 5,
    ("steam", "solid"): 1, ("steam", "gas"): 9, ("steam", "water"): 5,
    ("solid", "gas"): 1, ("solid", "water"): 3, ("gas", "water"): 3,
}

a) Build the symmetric co-occurrence matrix $X$ from these counts. Using $\mathrm{P}\left(w_t \mid w_g\right) = X_{g,t}/\sum_k X_{g,k}$, compute the three ratios above. Do they reproduce the large / small / near-one pattern Chapter 23 describes?

b) Implement the GloVe loss from Chapter 23,
$$\mathcal{L}_{\text{GloVe}} = \sum_{i,j} h(X_{ij})\left(\mathbf{v}_i^\top \mathbf{u}_j + b_i + c_j - \log X_{ij}\right)^2,$$
with weighting $h(x) = (x/x_{\max})^{\alpha}$ for $x < x_{\max}$ and $1$ otherwise. Because these toy counts are small, use $x_{\max} = 5$ and $\alpha = 0.75$ (the chapter's $x_{\max} = 100$ is tuned for a billion-word corpus). Initialize $\mathbf{v}_i, \mathbf{u}_j \in \mathbb{R}^3$ with `np.random.seed(26)` and $b_i = c_j = 0$, then run gradient descent over the pairs with $X_{ij} > 0$ for 800 epochs, printing the average weighted squared error periodically. Confirm it decreases.

c) Using $\mathbf{v}_i + \mathbf{u}_i$ as word $i$'s embedding, compute the cosine similarities between (ice, solid), (steam, solid), (ice, gas), and (steam, gas). Does the trained geometry recover the ratio asymmetry: ice aligned with solid, steam aligned with gas?

## Exercise 23.9: Self-Attention from Scratch

Four tokens, "dog", "puppy", "car", "vehicle", are embedded by hand (not learned) in $\mathbb{R}^3$ using three features: how strongly the word denotes an animal, a filler feature that carries no information about that distinction, and how strongly the word denotes a vehicle.

In [ ]:
import numpy as np

tokens = ["dog", "puppy", "car", "vehicle"]
X = np.array([
    [1.0, 0.5, 0.0],
    [0.9, 0.3, 0.0],
    [0.0, 0.4, 1.0],
    [0.0, 0.6, 0.9],
])

The projection matrices are fixed (not learned), so the whole computation can be checked by hand:
$$
W_Q = W_K = \begin{bmatrix} 1 & 0 \\ 0 & 0 \\ 0 & 1 \end{bmatrix}, \qquad
W_V = \begin{bmatrix} 0 & 1 \\ 1 & 1 \\ 1 & 0 \end{bmatrix}.
$$

a) Form $Q = XW_Q$, $K = XW_K$, $V = XW_V$ in numpy. What do $W_Q$ and $W_K$ do to the middle (filler) feature of $X$?

b) Compute the scaled scores $QK^\top/\sqrt{d_k}$ and the attention weight matrix $\alpha = \text{softmax}(QK^\top/\sqrt{d_k})$, with the softmax taken row-wise as in Chapter 23. Verify numerically that every row of $\alpha$ sums to 1.

c) For each token, report which other token it attends to most strongly. Does the attention pattern recover the fact that "dog" and "puppy" are related, and separately that "car" and "vehicle" are related, given that this was only implicit in $X$?

## Exercise 23.10: Byte Pair Encoding from Scratch

Chapter 23 introduces BPE on the word frequencies {"fast": 4, "faster": 3, "tall": 5, "taller": 4} and predicts that the first merges will be "t" + "a" $\rightarrow$ "ta", then "ta" + "l" $\rightarrow$ "tal". This exercise implements BPE and checks that prediction against real merge counts.

In [ ]:
from collections import defaultdict

corpus = {"fast": 4, "faster": 3, "tall": 5, "taller": 4}
vocab = {tuple(word) + ("</w>",): freq for word, freq in corpus.items()}

Each word starts out as a tuple of single-character symbols followed by an end-of-word marker `</w>`, tagged with its corpus frequency.

a) Write `get_pair_counts(vocab)`, returning for every adjacent symbol pair the total frequency (summed over words) with which it occurs, and `merge_pair(pair, vocab)`, which replaces every adjacent occurrence of `pair` with the merged symbol throughout `vocab`. Using these two functions, run 8 rounds of "find the most frequent pair, merge it," printing the merge and its count at each step. Do the first three merges match the chapter's prediction?

b) Print `vocab` after the 8 merges. How many merges does it take before "fast" and "tall" each become a single symbol (plus `</w>`)?

c) Write `bpe_tokenize(word, merges)`, which applies the learned merges, in order, to a new word, and use it to tokenize the unseen words "fastest" and "tallest". Why does the stem survive as one token while the superlative suffix does not, and what does this predict for a word like "smartest" if training data only ever showed the comparative "smarter"?

## Exercise 23.11: Masked Language Modeling from Scratch

Chapter 23 introduces BERT's masked language modeling (MLM): 15% of token positions are selected for prediction, and each selected position is corrupted by the 80/10/10 recipe (80% replaced with `[MASK]`, 10% replaced with a random word, 10% left unchanged). The loss
$$\mathcal{L}_{\text{MLM}} = -\sum_{m \in \mathcal{M}} \log P(x_m \mid \mathbf{x}_{\backslash \mathcal{M}})$$
is evaluated only at the selected positions $\mathcal{M}$. This exercise implements the masking and a simplified MLM objective on a toy corpus of twelve five-word sentences over a 16-word vocabulary.

In [ ]:
corpus = [
    "the king rules the kingdom", "the queen rules the kingdom",
    "the king loves the queen",   "the queen loves the king",
    "the king guards the castle",  "the queen guards the castle",
    "the dog chases the ball",     "the cat chases the ball",
    "the dog sleeps on mat",       "the cat sleeps on mat",
    "the king sleeps on throne",   "the dog guards the castle",
]
sentences = [s.split() for s in corpus]
vocab = sorted(set(w for s in sentences for w in s))
word2id = {w: i for i, w in enumerate(vocab)}
Vsz = len(vocab)
MASK = Vsz  # id reserved for the [MASK] token

a) Implement BERT's masking from Chapter 23: visit every token position, select it with probability 0.15, and for each selected position draw a corruption type with probabilities 0.8 (replace with `[MASK]`), 0.1 (replace with a random vocabulary word), 0.1 (leave unchanged). With `np.random.seed(26)`, report how many of the 60 positions are selected and the count of each corruption type, recording for every selected position the true word (needed for the loss).

b) Implement a simplified MLM loss. Give each vocabulary word and `[MASK]` a trainable $d = 8$ embedding. For a selected position $m$, form the context vector as the average of the (corrupted) embeddings at the other positions of the same sentence, project it to logits over the vocabulary with a trainable linear layer, and compute cross-entropy $-\log P(x_m \mid \mathbf{x}_{\backslash \mathcal{M}})$ only at the selected positions. Run gradient descent for 400 epochs, printing the average loss periodically. Confirm it decreases.

c) One selected position falls in the 10% "leave unchanged" branch: its input token is not altered, yet the loss is still computed there. Explain why BERT keeps these positions in the loss instead of scoring only positions where the input was actually replaced.